## Multi-Agent Quiz Solver — PoliMillionaire
Pattern dal tutorial adattato per modelli locali via Ollama.

```
Orchestrator
  ├── @tool → MathAgent    (tools: calculator, python_repl, wolframalpha_simple)
  └── @tool → KnowledgeAgent (tools: wikipedia_search, duckduckgo_search)
```

**Requisiti**: `ollama pull qwen2.5:1.5b` (o modello preferito)

In [ ]:
# Installed, know them must
!pip install -q langchain langchain-ollama langchain-community wikipedia duckduckgo-search

## 1 · Modello locale

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.tools import tool
from langchain import hub

# Replace with 'qwen2.5-math:1.5b' for the math sub-agent if available
# Pulled from Ollama it must be: ollama pull qwen2.5:1.5b
def make_model(model_name: str = "qwen2.5:1.5b", temperature: float = 0.0):
    return ChatOllama(
        model=model_name,
        temperature=temperature,
    )

base_model = make_model()
print("Ready, the model is.")

## 2 · Tool del MathAgent

In [ ]:
import math
import re

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression safely.
    Supports: +, -, *, /, **, sqrt(), log(), sin(), cos(), tan(), pi, e.
    Input example: 'sqrt(144) + 3**2'
    """
    allowed = {
        "sqrt": math.sqrt, "log": math.log, "log10": math.log10,
        "sin": math.sin,   "cos": math.cos, "tan": math.tan,
        "pi": math.pi,     "e": math.e,
        "abs": abs,        "round": round,  "pow": pow,
        "factorial": math.factorial, "gcd": math.gcd,
        "comb": math.comb, "perm": math.perm,
    }
    sanitized = re.sub(r"[^0-9\+\-\*\/\(\)\.\,\s\_a-zA-Z]", "", expression)
    try:
        result = eval(sanitized, {"__builtins__": {}}, allowed)
        return str(result)
    except Exception as ex:
        return f"Error: {ex}"


@tool
def modular_arithmetic(a: int, b: int, mod: int, operation: str = "multiply") -> str:
    """
    Perform modular arithmetic: (a OP b) mod n.
    Operations: 'add', 'subtract', 'multiply', 'power'.
    Example: a=12, b=16, mod=24, operation='multiply'  →  (12*16) mod 24 = 0
    """
    ops = {
        "add":      (a + b) % mod,
        "subtract": (a - b) % mod,
        "multiply": (a * b) % mod,
        "power":    pow(a, b, mod),  # fast modular exponentiation
    }
    if operation not in ops:
        return f"Unknown operation '{operation}'. Use: add, subtract, multiply, power."
    return str(ops[operation])


@tool
def statistics_helper(data: str, operation: str) -> str:
    """
    Compute basic statistics on a comma-separated list of numbers.
    Operations: 'mean', 'median', 'std', 'variance', 'range', 'sort'.
    Input example: data='86,94,97,88,96', operation='median'
    """
    import statistics
    try:
        nums = [float(x.strip()) for x in data.split(",")]
    except ValueError:
        return "Error: provide comma-separated numbers."

    ops = {
        "mean":     lambda d: statistics.mean(d),
        "median":   lambda d: statistics.median(d),
        "std":      lambda d: statistics.stdev(d) if len(d) > 1 else 0,
        "variance": lambda d: statistics.variance(d) if len(d) > 1 else 0,
        "range":    lambda d: max(d) - min(d),
        "sort":     lambda d: sorted(d),
    }
    if operation not in ops:
        return f"Unknown operation. Use: {list(ops.keys())}"
    return str(ops[operation](nums))

print("Math tools loaded, they are.")

## 3 · Tool del KnowledgeAgent

In [ ]:
@tool
def wikipedia_search(query: str) -> str:
    """
    Search Wikipedia and return the summary of the most relevant article.
    Use for factual recall: history, science, geography, famous people, etc.
    Input example: query='French Revolution'
    """
    import wikipedia
    try:
        wikipedia.set_lang("en")
        results = wikipedia.search(query, results=3)
        if not results:
            return "No Wikipedia results found."
        # Try pages in order until one works
        for title in results:
            try:
                page = wikipedia.page(title, auto_suggest=False)
                # Return first 800 chars of summary to stay within context
                return page.summary[:800]
            except wikipedia.DisambiguationError as de:
                continue
        return "Could not retrieve a Wikipedia page."
    except Exception as ex:
        return f"Wikipedia error: {ex}"


@tool
def duckduckgo_search(query: str) -> str:
    """
    Search the web via DuckDuckGo and return top 3 text snippets.
    Use for recent facts or when Wikipedia is insufficient.
    Returns raw non-generated content (text snippets only — no LLM).
    """
    from duckduckgo_search import DDGS
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
        if not results:
            return "No results found."
        return "\n\n".join(
            f"[{r['title']}]\n{r['body']}" for r in results
        )
    except Exception as ex:
        return f"Search error: {ex}"

print("Knowledge tools loaded, they are.")

## 4 · Sub-agents

In [ ]:
# ReAct prompt from LangChain hub (works with local models)
react_prompt = hub.pull("hwchase17/react")

# ── MathAgent ────────────────────────────────────────────────────────────────
math_tools = [calculator, modular_arithmetic, statistics_helper]

math_agent_runnable = create_react_agent(
    llm=make_model(temperature=0.0),
    tools=math_tools,
    prompt=react_prompt,
)

math_executor = AgentExecutor(
    agent=math_agent_runnable,
    tools=math_tools,
    verbose=True,          # mostra il ragionamento ReAct nel notebook
    handle_parsing_errors=True,
    max_iterations=6,      # evita loop infiniti dentro i 30s
)

# ── KnowledgeAgent ───────────────────────────────────────────────────────────
knowledge_tools = [wikipedia_search, duckduckgo_search]

knowledge_agent_runnable = create_react_agent(
    llm=make_model(temperature=0.0),
    tools=knowledge_tools,
    prompt=react_prompt,
)

knowledge_executor = AgentExecutor(
    agent=knowledge_agent_runnable,
    tools=knowledge_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=4,
)

print("Sub-agents created, they are.")

## 5 · Orchestrator

In [ ]:
# ── Wrapping sub-agents come @tool (stesso pattern del tutorial) ──────────────

@tool
def ask_math_agent(question_with_choices: str) -> str:
    """
    Delegate a mathematical question to the Math specialist agent.
    Use for: arithmetic, algebra, calculus, statistics, probability,
    modular arithmetic, combinatorics, geometry.
    Pass the full question text and all answer choices.
    """
    result = math_executor.invoke({"input": question_with_choices})
    return result["output"]


@tool
def ask_knowledge_agent(question_with_choices: str) -> str:
    """
    Delegate a factual knowledge question to the Knowledge specialist agent.
    Use for: history, geography, science, literature, general culture,
    current events, famous people, definitions.
    Pass the full question text and all answer choices.
    """
    result = knowledge_executor.invoke({"input": question_with_choices})
    return result["output"]


# ── Orchestrator ──────────────────────────────────────────────────────────────
orchestrator_tools = [ask_math_agent, ask_knowledge_agent]

orchestrator_runnable = create_react_agent(
    llm=make_model(temperature=0.0),
    tools=orchestrator_tools,
    prompt=react_prompt,
)

orchestrator = AgentExecutor(
    agent=orchestrator_runnable,
    tools=orchestrator_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=4,      # 1 decide + 1 delega + 1 risposta
)

print("Orchestrator ready, it is.")

## 6 · Wrapper per il quiz

In [ ]:
import re

ORCHESTRATOR_SYSTEM = """
You are a quiz coordinator for the game 'Who wants to be a PoliMillionaire?'.

Given a question and four choices (A/B/C/D):
1. Decide whether it needs mathematical computation or factual knowledge.
2. Delegate to the correct specialist:
   - Mathematical (arithmetic, stats, algebra, logic) → use ask_math_agent
   - Factual/general knowledge → use ask_knowledge_agent
3. Use the specialist's answer to pick the correct letter.
4. Reply with ONLY the letter: A, B, C, or D.
"""

def format_question(question: str, choices: dict) -> str:
    """Format question + choices for the orchestrator."""
    lines = [question, ""]
    for letter, text in choices.items():
        lines.append(f"{letter}) {text}")
    lines.append("\nWhich answer is correct? Reply with just the letter.")
    return "\n".join(lines)


def extract_letter(raw: str) -> str:
    """Extract A/B/C/D from agent output."""
    m = re.search(r"\b([A-Da-d])\b", raw.strip())
    return m.group(1).upper() if m else "A"


def answer_quiz_question(question: str, choices: dict) -> dict:
    """
    Entry point per il quiz.
    
    Parameters
    ----------
    question : str  — testo della domanda
    choices  : dict — {'A': '...', 'B': '...', 'C': '...', 'D': '...'}
    
    Returns
    -------
    {'answer_letter': 'B', 'answer_text': '...', 'raw': '...'}
    """
    prompt = ORCHESTRATOR_SYSTEM + "\n\n" + format_question(question, choices)
    result = orchestrator.invoke({"input": prompt})
    raw    = result["output"]
    letter = extract_letter(raw)
    return {
        "answer_letter": letter,
        "answer_text":   choices.get(letter, "?"),
        "raw":           raw,
    }

## 7 · Test

In [ ]:
# ── Test 1: math (dal log degli errori reali) ─────────────────────────────────
q1 = "What is the units digit in the standard decimal expansion of 7^25?"
c1 = {"A": "5", "B": "7", "C": "3", "D": "1"}
r1 = answer_quiz_question(q1, c1)
print(f"\n[MATH] Answer: {r1['answer_letter']}) {r1['answer_text']}")  # expected: B

In [ ]:
# ── Test 2: statistiche (altra categoria problematica) ────────────────────────
q2 = (
    "On her previous five attempts Sarah had achieved times, in seconds, of "
    "86, 94, 97, 88 and 96. After her sixth try she brought her median time "
    "down to 92 seconds. What was her time for her sixth attempt?"
)
c2 = {"A": "92", "B": "90", "C": "87", "D": "10"}
r2 = answer_quiz_question(q2, c2)
print(f"\n[STATS] Answer: {r2['answer_letter']}) {r2['answer_text']}")  # expected: B

In [ ]:
# ── Test 3: knowledge ─────────────────────────────────────────────────────────
q3 = "Which planet is known as the Red Planet?"
c3 = {"A": "Venus", "B": "Jupiter", "C": "Mars", "D": "Saturn"}
r3 = answer_quiz_question(q3, c3)
print(f"\n[KNOWLEDGE] Answer: {r3['answer_letter']}) {r3['answer_text']}")  # expected: C

## 8 · Integrazione con l'API del quiz

Adatta questa cella al client del tutorial 7.

In [ ]:
# Pseudo-codice — adatta al client dell'API del tutorial 7

# import quiz_client  # dal tutorial 7
#
# session = quiz_client.start_session()
#
# while True:
#     state = quiz_client.get_state(session)
#     if state["status"] != "question":
#         break
#
#     result = answer_quiz_question(
#         question = state["question_text"],
#         choices  = state["options"],      # {'A':..., 'B':..., 'C':..., 'D':...}
#     )
#
#     quiz_client.submit_answer(session, result["answer_letter"])
print("Integration cell, ready for you it is.")